In [17]:
import time
import threading
from datetime import datetime, timedelta
from collections import deque
import pandas as pd

# Tiempo durante el que se cuentan los intentos de login
VENTANA_SEGUNDOS = 10

# Número de intentos fallidos necesarios para banear una IP
MAX_INTENTOS = 5

# Tiempo que permanece baneada una IP
TIEMPO_BANEO = 30

In [18]:
# Cada evento representa una petición que recibe el servidor
def crear_evento(ip, endpoint, es_exitoso=True):
    return {
        "timestamp": datetime.now(),
        "ip": ip,
        "endpoint": endpoint,
        "es_exitoso": es_exitoso
    }

In [19]:
# Creamos varios eventos para probar el funcionamiento del WAF
def generar_eventos():
    eventos = []

    # Peticiones normales de distintos usuarios
    eventos.append(crear_evento("192.168.1.20", "/api/login"))
    eventos.append(crear_evento("192.168.1.20", "/api/login"))
    eventos.append(crear_evento("192.168.1.15", "/api/transferencias"))

    # Simulamos cinco intentos fallidos desde la misma IP
    for _ in range(5):
        eventos.append(
            crear_evento("203.0.113.45", "/api/login", False)
        )

    # La misma IP sigue haciendo peticiones después del baneo
    eventos.append(crear_evento("203.0.113.45", "/api/login", False))
    eventos.append(crear_evento("203.0.113.45", "/api/transferencias"))

    # Otra petición normal para comprobar que el WAF sigue funcionando
    eventos.append(crear_evento("192.168.1.10", "/api/login"))

    return eventos


# El productor va introduciendo los eventos en la cola
def productor(cola, estado):
    print("Productor iniciado...")

    for evento in generar_eventos():
        cola.append(evento)
        time.sleep(0.5)

    # Indicamos que ya se han generado todos los eventos
    estado["finalizado"] = True

    print("Productor finalizado.")

In [20]:
class WAF:

    def __init__(self):
        # Guardamos los intentos fallidos de cada IP
        self.intentos_fallidos = {}

        # Guardamos las IP baneadas y cuándo termina su baneo
        self.ips_baneadas = {}

    # Comprobamos si una IP sigue estando baneada
    def esta_baneada(self, ip, ahora):

        if ip in self.ips_baneadas:
            if ahora < self.ips_baneadas[ip]:
                return True
            else:
                # Si el baneo ha terminado, eliminamos la IP
                del self.ips_baneadas[ip]

        return False

    # Recibimos un evento y decidimos si se permite o se bloquea
    def procesar_evento(self, evento):

        ip = evento["ip"]
        endpoint = evento["endpoint"]
        ahora = evento["timestamp"]

        # Si la IP está baneada, no dejamos pasar la petición
        if self.esta_baneada(ip, ahora):
            return "BLOQUEADO"

        # Las peticiones que no son de login se permiten
        if endpoint != "/api/login":
            return "PERMITIDO"

        # Si el login ha sido correcto, no contamos el intento
        if evento["es_exitoso"]:
            return "PERMITIDO"

        # Si es la primera vez que vemos esta IP, creamos su lista
        if ip not in self.intentos_fallidos:
            self.intentos_fallidos[ip] = deque()

        intentos = self.intentos_fallidos[ip]

        # Calculamos el límite de tiempo de la ventana
        limite = ahora - timedelta(seconds=VENTANA_SEGUNDOS)

        # Quitamos los intentos que han quedado fuera de los 10 segundos
        while intentos and intentos[0] < limite:
            intentos.popleft()

        # Añadimos el nuevo intento fallido
        intentos.append(ahora)

        # Si alcanza el número máximo de intentos, baneamos la IP
        if len(intentos) >= MAX_INTENTOS:
            self.ips_baneadas[ip] = ahora + timedelta(seconds=TIEMPO_BANEO)

            print(f"IP {ip} baneada.")

            return "BANEADO"

        return "PERMITIDO"

In [21]:
class ConsumidorEstadisticas:

    def __init__(self):
        # Aquí guardamos los eventos después de pasar por el WAF
        self.eventos = []

    # Registramos cada evento junto con el resultado del WAF
    def registrar_evento(self, evento, resultado):

        self.eventos.append({
            "timestamp": evento["timestamp"],
            "ip": evento["ip"],
            "endpoint": evento["endpoint"],
            "resultado": resultado
        })

        # Mostramos por pantalla qué ha ocurrido con cada petición
        print(
            f"[{evento['timestamp'].strftime('%H:%M:%S')}] "
            f"IP: {evento['ip']} | "
            f"Endpoint: {evento['endpoint']} | "
            f"Resultado: {resultado}"
        )

    # Calculamos un resumen de todos los eventos recibidos
    def mostrar_resumen(self):

        df = pd.DataFrame(self.eventos)

        total = len(df)
        permitidos = len(df[df["resultado"] == "PERMITIDO"])

        bloqueados = len(
            df[df["resultado"].isin(["BANEADO", "BLOQUEADO"])]
        )

        # Obtenemos las IP que han sido detectadas por el WAF
        ips_detectadas = df[
            df["resultado"].isin(["BANEADO", "BLOQUEADO"])
        ]["ip"].unique()

        print("\nResumen del WAF")
        print(f"Total de eventos: {total}")
        print(f"Eventos permitidos: {permitidos}")
        print(f"Eventos bloqueados: {bloqueados}")
        print(f"IPs detectadas: {list(ips_detectadas)}")

        # Agrupamos los resultados según el endpoint y el estado
        print("\nEventos agrupados por endpoint y resultado:")

        agrupados = df.groupby(
            ["endpoint", "resultado"]
        ).size().unstack(fill_value=0)

        print(agrupados)

In [22]:
# Creamos la cola donde el productor irá dejando los eventos
cola = deque()

# Esta variable indica si el productor ha terminado
estado = {"finalizado": False}

# Creamos el WAF y el consumidor
waf = WAF()
consumidor = ConsumidorEstadisticas()

# Iniciamos el productor en un hilo independiente
hilo_productor = threading.Thread(
    target=productor,
    args=(cola, estado)
)

hilo_productor.start()

print("\nIniciando procesamiento de eventos...\n")

# El consumidor sigue trabajando mientras queden eventos
# o el productor todavía no haya terminado
while not estado["finalizado"] or cola:

    if cola:
        evento = cola.popleft()

        # El WAF comprueba la petición
        resultado = waf.procesar_evento(evento)

        # Guardamos el resultado para las estadísticas
        consumidor.registrar_evento(evento, resultado)

    else:
        # Esperamos un poco si todavía no hay eventos disponibles
        time.sleep(0.1)

# Esperamos a que termine el hilo del productor
hilo_productor.join()

# Mostramos las estadísticas finales
consumidor.mostrar_resumen()

Productor iniciado...

Iniciando procesamiento de eventos...

[15:19:14] IP: 192.168.1.20 | Endpoint: /api/login | Resultado: PERMITIDO
[15:19:14] IP: 192.168.1.20 | Endpoint: /api/login | Resultado: PERMITIDO
[15:19:14] IP: 192.168.1.15 | Endpoint: /api/transferencias | Resultado: PERMITIDO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: PERMITIDO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: PERMITIDO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: PERMITIDO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: PERMITIDO
IP 203.0.113.45 baneada.
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: BANEADO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/login | Resultado: BLOQUEADO
[15:19:14] IP: 203.0.113.45 | Endpoint: /api/transferencias | Resultado: BLOQUEADO
[15:19:14] IP: 192.168.1.10 | Endpoint: /api/login | Resultado: PERMITIDO
Productor finalizado.

Resumen del WAF
Total de eventos: 11
Eventos permitidos: 8
E